# GAN evaluation (FID, Inception Score) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normal_pdf(x,mu,sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*np.sqrt(2*np.pi))
def softmax(a):
    a=np.asarray(a,dtype=float); e=np.exp(a-a.max()); return e/e.sum()
def mse(a,b): return float(np.mean((np.asarray(a)-np.asarray(b))**2))
print('setup ok')

## distributional sample metrics
Probability, covariance, and KL divergence are the prerequisites. FID compares feature means and covariances; Inception Score compares confident conditional labels to diverse marginal labels. These metrics become cautionary tools for diffusion (10.12) and text-to-image systems (10.14).

In [ ]:
# 1) toy FID compares feature means and variances
m1=np.array([0.,0.]); m2=np.array([1.,.5]); v1=np.array([1.,.5]); v2=np.array([.6,.8]); fid=np.sum((m1-m2)**2)+np.sum(v1+v2-2*np.sqrt(v1*v2))
plt.figure(figsize=(4,3)); plt.bar(['mean gap','cov gap','FID'],[np.sum((m1-m2)**2),np.sum(v1+v2-2*np.sqrt(v1*v2)),fid]); plt.title('FID pieces')
assert abs(fid-1.3358956)<1e-5

In [ ]:
# 2) FID worsens as generated mean shifts
shifts=np.linspace(0,3,80); fids=shifts**2
plt.figure(figsize=(5,3)); plt.plot(shifts,fids); plt.xlabel('mean shift'); plt.ylabel('FID'); plt.title('mean mismatch raises FID')
assert fids[-1]>fids[0]

In [ ]:
# 3) covariance mismatch matters even with equal means
scales=np.linspace(.2,2,80); covgap=1+scales-2*np.sqrt(scales)
plt.figure(figsize=(5,3)); plt.plot(scales,covgap); plt.xlabel('generated variance'); plt.title('FID covariance term')
assert covgap[0]>min(covgap)

In [ ]:
# 4) Inception Score rewards confident class predictions
p=softmax([1.2,.3,-.4]); u=np.ones(3)/3; score=math.exp(np.sum(p*(np.log(p)-np.log(u))))
plt.figure(figsize=(4,3)); plt.bar(['class0','class1','class2'],p); plt.title(f'IS contribution {score:.2f}')
assert score>1

In [ ]:
# 5) missing modes can have confident but low-diversity predictions
preds=np.array([[.9,.05,.05],[.9,.05,.05],[.9,.05,.05]])
marg=preds.mean(axis=0); div=-np.sum(marg*np.log(marg))
plt.figure(figsize=(4,3)); plt.bar(range(3),marg); plt.title('collapsed marginal has low diversity')
assert marg[0]>0.8 and div<math.log(3)